In [34]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

demo_context = {"workspace_id": "acm", "user_id": "gqt"}


def get_namespace_path_from_context(context):
    return (
        "memory",
        context["workspace_id"],
        context["user_id"],
    )


def get_namespace_from_runtime(runtime):
    return get_namespace_path_from_context(runtime.context)

In [35]:
from deepagents.backends.utils import create_file_data

store.put(
    get_namespace_path_from_context(demo_context),
    "/AGENTS.md",
    create_file_data("""\
# 项目规范

## 代码风格
- 所有函数都必须有类型注解
- 字符串格式化请使用 f-string
- 最大行长 88 个字符
- 文件操作请使用 `pathlib.Path`，而不是 `os.path`

## 工作流
- 用以下命令运行测试：`uv run pytest`
- CI 流水线会在每次推送到 `main` 时运行
- 尽早打开草稿 PR，这样评审者可以一路跟进
"""),
)

In [36]:
print(store.list_namespaces())
a = store.search(('memory', 'acm', 'gqt'))
from rich import print as rp

rp(a)

[('memory', 'acm', 'gqt')]


[
    Item(namespace=['memory', 'acm', 'gqt'], key='/AGENTS.md', value={'content': '# 项目规范\n\n## 代码风格\n- 
所有函数都必须有类型注解\n- 字符串格式化请使用 f-string\n- 最大行长 88 个字符\n- 文件操作请使用 
`pathlib.Path`，而不是 `os.path`\n\n## 工作流\n- 用以下命令运行测试：`uv run pytest`\n- CI 流水线会在每次推送到 
`main` 时运行\n- 尽早打开草稿 PR，这样评审者可以一路跟进\n', 'encoding': 'utf-8', 'created_at': 
'2026-09-03T03:27:36.725993+00:00', 'modified_at': '2026-09-03T03:27:36.725993+00:00'}, 
created_at='2026-09-03T03:27:36.726008+00:00', updated_at='2026-09-03T03:27:36.726008+00:00', score=None)
]

In [37]:
from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend

from models import model

agent = create_deep_agent(
    model=model,
    store=store,
    backend=CompositeBackend(
        default=StateBackend(),
        routes={
            "/memories/": StoreBackend(namespace=get_namespace_from_runtime)
        }
    ),
    memory=["/memories/AGENTS.md"],
    system_prompt="你是这个项目的一名乐于助人的编程助手。",
)

In [38]:
result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "在这个项目中，我应该用什么工具来处理文件路径？"}
        ]
    },
    context=demo_context,
)
print("--- 问题 1 ---")
print(result["messages"][-1].content)

--- 问题 1 ---
根据这个项目的规范（`AGENTS.md`），处理文件路径应该使用 **`pathlib.Path`**，而不是 `os.path`。

例如：

```python
from pathlib import Path

# 推荐
data_dir = Path("data")
output = data_dir / "result.txt"
output.write_text("hello")
```

而不是：

```python
import os.path  # ❌ 不推荐

path = os.path.join("data", "result.txt")
```

另外两点相关规范：
- 所有函数必须有类型注解
- 字符串格式化用 f-string，最大行长 88 个字符


In [39]:
result2 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "记住：团队改用 ruff 来做 lint 了。请更新你的记忆。",
            }
        ]
    },
    context=demo_context,
)
print("\n--- 问题 2 ---")
print(result2["messages"][-1].content)


--- 问题 2 ---
已更新记忆：在“代码风格”下新增了一条“团队用 ruff 来做 lint”。

顺带一个小建议：如果方便的话，可以再补充一句具体的运行方式（比如 `uv run ruff check .`）和 CI 中对应的检查环节，这样我以后写代码时能照着同样的命令自查。需要的话我也可以帮你加上。


In [40]:
a = store.search(('memory', 'acm', 'gqt'))
rp(a)

[
    Item(namespace=['memory', 'acm', 'gqt'], key='/AGENTS.md', value={'content': '# 项目规范\n\n## 代码风格\n- 
所有函数都必须有类型注解\n- 字符串格式化请使用 f-string\n- 最大行长 88 个字符\n- 文件操作请使用 
`pathlib.Path`，而不是 `os.path`\n- 团队用 ruff 来做 lint\n\n## 工作流\n- 用以下命令运行测试：`uv run pytest`\n- CI
流水线会在每次推送到 `main` 时运行\n- 尽早打开草稿 PR，这样评审者可以一路跟进\n', 'encoding': 'utf-8', 
'created_at': '2026-09-03T03:27:36.725993+00:00', 'modified_at': '2026-09-03T03:27:43.631994+00:00'}, 
created_at='2026-09-03T03:27:43.632043+00:00', updated_at='2026-09-03T03:27:43.632044+00:00', score=None)
]